# Stage 5 — Final Holdout Model Comparison

This notebook performs the one-time Holdout discrimination comparison only after mechanically verifying the complete Pre-Holdout Freeze Record produced by Notebook 11. It contains no tuning, search, feature selection, preprocessing redesign, boosting-round selection, calibration, threshold optimization, DeLong test, SHAP work, or post-Holdout model change.


In [1]:
from pathlib import Path
import hashlib
import importlib.metadata as metadata
import json
import re
import sys

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda value: f"{value:.12f}")

PROJECT_ROOT = Path("/Users/yu/2.工作/Risk Analysis Project")
TARGET = "default.payment.next.month"
ID_COL = "ID"
RANDOM_STATE = 42
EXPECTED_TRAIN_HASH = "bcb37af63ae7005df78e6140837594b5d650d27148b46b5d51354430ad958857"
EXPECTED_TEST_HASH = "d6748249f9f9fb2855746e0ed980b5013047267104090a14aa8138a700fe6f0f"

FINAL_HYPERPARAMETERS = {
    "objective": "binary:logistic", "booster": "gbtree", "eval_metric": "auc",
    "tree_method": "hist", "eta": 0.10, "max_depth": 4, "min_child_weight": 1.0,
    "subsample": 0.70, "colsample_bytree": 0.70, "gamma": 0.0,
    "reg_alpha": 0.0, "reg_lambda": 1.0, "scale_pos_weight": 1.0,
    "seed": 42, "nthread": 4, "verbosity": 0,
}
FINAL_BEST_ROUND = 54
FINAL_DEVELOPMENT_MEAN_AUC = 0.785879288351590
FINAL_DEVELOPMENT_AUC_SD = 0.003556829920052
FINAL_DEVELOPMENT_MEAN_KS = 0.437782987717489
FINAL_DEVELOPMENT_KS_SD = 0.009842401657479

development_notebook = PROJECT_ROOT / "notebooks/11_xgboost_challenger_development.ipynb"
development_nb = json.loads(development_notebook.read_text(encoding="utf-8"))
output_text = "\n".join(
    "".join(output.get("text", []))
    for cell in development_nb["cells"]
    for output in cell.get("outputs", [])
    if "text" in output
)
match = re.search(r"STAGE5_DEVELOPMENT_RESULT_JSON=(\{.*\})", output_text)
assert match is not None
freeze_source = json.loads(match.group(1))
freeze_record = freeze_source["pre_holdout_freeze_record"]
assert freeze_source["status"] == "CHALLENGER_FROZEN_READY_FOR_HOLDOUT"
assert freeze_source["holdout_predictively_accessed"] is False
assert freeze_record["freeze_status"] == "FROZEN BEFORE HOLDOUT PREDICTIVE ACCESS"
assert freeze_source["final_joint_count"] == 675
assert freeze_source["final_best_round"] == FINAL_BEST_ROUND
assert freeze_source["final_hyperparameters"] == FINAL_HYPERPARAMETERS
assert freeze_source["train_hash"] == EXPECTED_TRAIN_HASH
assert freeze_source["test_hash"] == EXPECTED_TEST_HASH
assert abs(freeze_source["final_selected"]["Mean AUC"] - FINAL_DEVELOPMENT_MEAN_AUC) <= 1e-15
assert abs(freeze_source["final_selected"]["AUC SD"] - FINAL_DEVELOPMENT_AUC_SD) <= 1e-15
assert Path(sys.executable).resolve() == (PROJECT_ROOT / ".venv/bin/python").resolve()
assert sys.version.split()[0] == "3.9.6" and xgb.__version__ == "2.1.4"
FREEZE_GATE_STATUS = "PASS — complete Notebook 11 Pre-Holdout Freeze Record verified before predictive Holdout access"
display(pd.DataFrame({"Frozen value": [
    FREEZE_GATE_STATUS, FINAL_HYPERPARAMETERS, FINAL_BEST_ROUND,
    FINAL_DEVELOPMENT_MEAN_AUC, FINAL_DEVELOPMENT_AUC_SD,
    FINAL_DEVELOPMENT_MEAN_KS, FINAL_DEVELOPMENT_KS_SD,
]}, index=[
    "Freeze gate", "Final hyperparameters", "Frozen best_round", "Development Mean CV AUC",
    "Development AUC sample SD", "Development Mean CV KS", "Development KS sample SD",
]))


,Frozen value
Freeze gate,PASS — complete Notebook 11 Pre-Holdout Freeze...
Final hyperparameters,"{'objective': 'binary:logistic', 'booster': 'g..."
Frozen best_round,54
Development Mean CV AUC,0.785879288352
Development AUC sample SD,0.003556829920
Development Mean CV KS,0.437782987717
Development KS sample SD,0.009842401657


## 1. Frozen split replay and technical identity check

The outer split is replayed exactly. Before the freeze gate above, no Holdout predictor or target is read. This cell performs only the established row/ID/hash checks; predictive access remains gated until both frozen models have been reproduced/refit below.


In [2]:
def stable_id_hash(values):
    canonical = "\n".join(str(int(value)) for value in sorted(values))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

raw_df = pd.read_csv(PROJECT_ROOT / "data/raw/UCI_Credit_Card.csv")
development_raw, holdout_raw = train_test_split(
    raw_df, test_size=0.20, random_state=RANDOM_STATE, stratify=raw_df[TARGET]
)
train_hash = stable_id_hash(development_raw[ID_COL])
test_hash = stable_id_hash(holdout_raw[ID_COL])
assert raw_df.shape == (30000, 25)
assert len(development_raw) == 24000 and len(holdout_raw) == 6000
assert set(development_raw[ID_COL]).isdisjoint(set(holdout_raw[ID_COL]))
assert set(development_raw[ID_COL]) | set(holdout_raw[ID_COL]) == set(raw_df[ID_COL])
assert train_hash == EXPECTED_TRAIN_HASH and test_hash == EXPECTED_TEST_HASH
print("PASS — frozen 24,000/6,000 split identity and disjoint/union coverage verified.")
print("Train ID SHA-256:", train_hash)
print("Test ID SHA-256:", test_hash)


PASS — frozen 24,000/6,000 split identity and disjoint/union coverage verified.
Train ID SHA-256: bcb37af63ae7005df78e6140837594b5d650d27148b46b5d51354430ad958857
Test ID SHA-256: d6748249f9f9fb2855746e0ed980b5013047267104090a14aa8138a700fe6f0f


## 2. Frozen feature preparation on Development only

The 26 conceptual predictors, three exclusions, four fixed education indicators, and 29-column order are reproduced from Stage 3. Holdout features are not transformed in this section.


In [3]:
PAY_COLS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
BILL_COLS = [f"BILL_AMT{i}" for i in range(1, 7)]
PAY_AMT_COLS = [f"PAY_AMT{i}" for i in range(1, 7)]
EDUCATION_CATEGORIES = ["Graduate School", "University", "High School", "Other / Unknown"]
EDUCATION_MAP = {1: "Graduate School", 2: "University", 3: "High School", 0: "Other / Unknown", 4: "Other / Unknown", 5: "Other / Unknown", 6: "Other / Unknown"}
CONCEPTUAL_PREDICTORS = [
    *PAY_COLS, "delinquent_months", "max_delinquency", "severe_delinquent_months",
    "EDUCATION_GROUP", "LIMIT_BAL", *BILL_COLS, *PAY_AMT_COLS,
    "current_utilization", "avg_utilization", "max_utilization",
]
EDUCATION_INDICATORS = [f"EDUCATION_GROUP__{c}" for c in EDUCATION_CATEGORIES]
ENCODED_COLUMNS = [
    *PAY_COLS, "delinquent_months", "max_delinquency", "severe_delinquent_months",
    *EDUCATION_INDICATORS, "LIMIT_BAL", *BILL_COLS, *PAY_AMT_COLS,
    "current_utilization", "avg_utilization", "max_utilization",
]
EXCLUDED_VARIABLES = ["SEX", "MARRIAGE_GROUP", "AGE"]
assert CONCEPTUAL_PREDICTORS == freeze_record["conceptual_predictors"]
assert ENCODED_COLUMNS == freeze_record["encoded_columns"]
assert EXCLUDED_VARIABLES == freeze_record["excluded_variables"]

def apply_frozen_stage3_features(raw_predictor_input):
    assert TARGET not in raw_predictor_input.columns
    work = raw_predictor_input.copy(deep=True)
    assert set(work["EDUCATION"].unique()).issubset(EDUCATION_MAP)
    assert (work["LIMIT_BAL"] > 0).all()
    work["EDUCATION_GROUP"] = work["EDUCATION"].map(EDUCATION_MAP)
    raw_pay = work[PAY_COLS]
    work["delinquent_months"] = (raw_pay > 0).sum(axis=1).astype("int64")
    work["max_delinquency"] = raw_pay.clip(lower=0).max(axis=1).astype("int64")
    work["severe_delinquent_months"] = (raw_pay >= 2).sum(axis=1).astype("int64")
    positive_bills = work[BILL_COLS].clip(lower=0)
    work["current_utilization"] = positive_bills["BILL_AMT1"] / work["LIMIT_BAL"]
    work["avg_utilization"] = positive_bills.mean(axis=1) / work["LIMIT_BAL"]
    work["max_utilization"] = positive_bills.max(axis=1) / work["LIMIT_BAL"]
    return work

def build_challenger_matrix(feature_frame):
    matrix = pd.DataFrame(index=feature_frame.index)
    for name in [*PAY_COLS, "delinquent_months", "max_delinquency", "severe_delinquent_months"]:
        matrix[name] = feature_frame[name]
    for category, column in zip(EDUCATION_CATEGORIES, EDUCATION_INDICATORS):
        matrix[column] = (feature_frame["EDUCATION_GROUP"] == category).astype("int8")
    matrix["LIMIT_BAL"] = feature_frame["LIMIT_BAL"]
    for name in [*BILL_COLS, *PAY_AMT_COLS, "current_utilization", "avg_utilization", "max_utilization"]:
        matrix[name] = feature_frame[name]
    assert matrix.columns.tolist() == ENCODED_COLUMNS and matrix.shape[1] == 29
    assert matrix[EDUCATION_INDICATORS].sum(axis=1).eq(1).all()
    assert matrix.notna().all().all() and np.isfinite(matrix.to_numpy(float)).all()
    return matrix.astype(float)

development_features = apply_frozen_stage3_features(development_raw.drop(columns=[TARGET]))
X_development = build_challenger_matrix(development_features)
y_development = development_raw[TARGET].astype(int)
assert X_development.shape == (24000, 29)
print("PASS — Development Challenger matrix shape=(24000, 29); no missing/non-finite values; Holdout not transformed.")


PASS — Development Challenger matrix shape=(24000, 29); no missing/non-finite values; Holdout not transformed.


## 3. Frozen Primary reproduction on Development

The Stage 4.7 ten-variable WOE Logistic specification is loaded without redevelopment. Frozen bins and Development-derived WOE mappings reproduce the accepted apparent AUC/KS before any Holdout transformation.


In [4]:
PRIMARY_PREDICTORS = ["PAY_0", "severe_delinquent_months", *PAY_AMT_COLS, "LIMIT_BAL", "avg_utilization"]
PRIMARY_PARAMETERS = pd.Series({
    "const": -1.254276175288, "PAY_0": -0.556303701580,
    "severe_delinquent_months": -0.513060045720,
    "PAY_AMT1": -0.193755569096, "PAY_AMT2": -0.222092687932,
    "PAY_AMT3": -0.245401438534, "PAY_AMT4": -0.165041793818,
    "PAY_AMT5": -0.097122437149, "PAY_AMT6": -0.164349101704,
    "LIMIT_BAL": -0.226160642201, "avg_utilization": -0.405621157758,
})
PAY_DOMAIN = set(range(-2, 9))
PAY0_MAP = {-2: "[-2,-1,0]", -1: "[-2,-1,0]", 0: "[-2,-1,0]", 1: "[1]", 2: "[2+]", 3: "[2+]", 4: "[2+]", 5: "[2+]", 6: "[2+]", 7: "[2+]", 8: "[2+]"}

def finite_number(value, name):
    numeric = float(value)
    if not np.isfinite(numeric): raise ValueError(f"{name}: non-finite input")
    return numeric
def map_pay_0(value):
    numeric = finite_number(value, "PAY_0"); raw = int(numeric)
    if raw != numeric or raw not in PAY_DOMAIN: raise ValueError("PAY_0 outside approved domain")
    return PAY0_MAP[raw]
def map_severe(value):
    numeric = finite_number(value, "severe"); raw = int(numeric)
    mapping = {0: "0", 1: "1", 2: "2", 3: "3", 4: "4+", 5: "4+", 6: "4+"}
    if raw != numeric or raw not in mapping: raise ValueError("severe outside approved domain")
    return mapping[raw]
def map_payment(value):
    numeric = finite_number(value, "PAY_AMT")
    if numeric < 0: raise ValueError("negative PAY_AMT")
    if numeric == 0: return "0"
    if numeric <= 5000: return "(0,5k]"
    if numeric <= 20000: return "(5k,20k]"
    return ">20k"
def map_limit(value):
    numeric = finite_number(value, "LIMIT_BAL")
    if numeric <= 30000: return "<=30k"
    if numeric <= 140000: return "(30k,140k]"
    if numeric <= 240000: return "(140k,240k]"
    return ">240k"
def map_util(value):
    numeric = finite_number(value, "avg_utilization")
    if numeric < 0: raise ValueError("negative utilization")
    if numeric == 0: return "0"
    if numeric <= .20: return "(0,0.2]"
    if numeric <= .60: return "(0.2,0.6]"
    if numeric <= 1: return "(0.6,1]"
    return ">1"

BIN_ORDERS = {
    "PAY_0": ["[-2,-1,0]", "[1]", "[2+]"],
    "severe_delinquent_months": ["0", "1", "2", "3", "4+"],
    **{name: ["0", "(0,5k]", "(5k,20k]", ">20k"] for name in PAY_AMT_COLS},
    "LIMIT_BAL": ["<=30k", "(30k,140k]", "(140k,240k]", ">240k"],
    "avg_utilization": ["0", "(0,0.2]", "(0.2,0.6]", "(0.6,1]", ">1"],
}
BIN_MAPPERS = {"PAY_0": map_pay_0, "severe_delinquent_months": map_severe, **{n: map_payment for n in PAY_AMT_COLS}, "LIMIT_BAL": map_limit, "avg_utilization": map_util}

development_bins = pd.DataFrame(index=development_features.index)
primary_woe_maps = {}
for predictor in PRIMARY_PREDICTORS:
    labels = development_features[predictor].map(BIN_MAPPERS[predictor])
    development_bins[predictor] = pd.Categorical(labels, categories=BIN_ORDERS[predictor], ordered=True)
    grouped = pd.DataFrame({"Bin": development_bins[predictor], TARGET: y_development}).groupby("Bin", observed=False)[TARGET].agg(Count="size", Bad="sum").reindex(BIN_ORDERS[predictor], fill_value=0)
    grouped["Good"] = grouped["Count"] - grouped["Bad"]
    assert (grouped[["Good", "Bad"]] > 0).all().all()
    primary_woe_maps[predictor] = np.log((grouped["Good"] / grouped["Good"].sum()) / (grouped["Bad"] / grouped["Bad"].sum())).to_dict()

def map_primary_woe(feature_frame):
    matrix = pd.DataFrame(index=feature_frame.index)
    for predictor in PRIMARY_PREDICTORS:
        labels = feature_frame[predictor].map(BIN_MAPPERS[predictor])
        matrix[predictor] = labels.map(primary_woe_maps[predictor]).astype(float)
    assert matrix.notna().all().all() and np.isfinite(matrix.to_numpy()).all()
    return matrix

primary_dev_woe = map_primary_woe(development_features)
primary_dev_lp = float(PRIMARY_PARAMETERS["const"]) + primary_dev_woe.to_numpy() @ PRIMARY_PARAMETERS[PRIMARY_PREDICTORS].to_numpy()
primary_dev_probability = 1 / (1 + np.exp(-primary_dev_lp))
def ks_metric(y_true, probability):
    fpr, tpr, thresholds = roc_curve(y_true, probability, pos_label=1)
    gaps = tpr - fpr; idx = int(np.argmax(gaps))
    return float(gaps[idx]), float(thresholds[idx])
primary_dev_auc = float(roc_auc_score(y_development, primary_dev_probability))
primary_dev_ks, _ = ks_metric(y_development, primary_dev_probability)
assert abs(primary_dev_auc - 0.772417097808) <= 1e-9
assert abs(primary_dev_ks - 0.430307867280) <= 1e-9
PRIMARY_FROZEN_STATUS = "PASS — frozen Stage 4.7 Primary parameters, bins, WOE convention, apparent AUC, and apparent KS reproduced"
print(PRIMARY_FROZEN_STATUS)


PASS — frozen Stage 4.7 Primary parameters, bins, WOE convention, apparent AUC, and apparent KS reproduced


## 4. Frozen Challenger full-Development refit

The single frozen Challenger is refit on all 24,000 Development rows for exactly 54 rounds. No validation split or early stopping is used.


In [5]:
ddevelopment = xgb.DMatrix(X_development, label=y_development, feature_names=ENCODED_COLUMNS)
frozen_challenger = xgb.train(FINAL_HYPERPARAMETERS, ddevelopment, num_boost_round=FINAL_BEST_ROUND, verbose_eval=False)
dev_check_probability = frozen_challenger.predict(ddevelopment)
assert len(dev_check_probability) == 24000 and np.isfinite(dev_check_probability).all()
CHALLENGER_FROZEN_STATUS = "PASS — frozen Challenger refit on all 24,000 Development rows for exactly 54 rounds without early stopping"
print(CHALLENGER_FROZEN_STATUS)


PASS — frozen Challenger refit on all 24,000 Development rows for exactly 54 rounds without early stopping


## 5. One-time Holdout opening and frozen-model comparison

Both models and preprocessing specifications are now frozen and verified. The same 6,000 Holdout observations are transformed once and scored once. No decision or model change follows from these results. KS thresholds are metric diagnostics only, not operational cutoffs.


In [6]:
assert FREEZE_GATE_STATUS.startswith("PASS")
assert PRIMARY_FROZEN_STATUS.startswith("PASS")
assert CHALLENGER_FROZEN_STATUS.startswith("PASS")
HOLDOUT_OPENING_STATUS = "OPENED ONCE AFTER BOTH MODELS FROZEN"

holdout_features = apply_frozen_stage3_features(holdout_raw.drop(columns=[TARGET]))
X_holdout = build_challenger_matrix(holdout_features)
y_holdout = holdout_raw[TARGET].astype(int)
assert X_holdout.shape == (6000, 29)
assert X_holdout.columns.tolist() == X_development.columns.tolist() == ENCODED_COLUMNS
assert y_holdout.index.equals(X_holdout.index)

primary_holdout_woe = map_primary_woe(holdout_features)
primary_holdout_lp = float(PRIMARY_PARAMETERS["const"]) + primary_holdout_woe.to_numpy() @ PRIMARY_PARAMETERS[PRIMARY_PREDICTORS].to_numpy()
primary_holdout_probability = 1 / (1 + np.exp(-primary_holdout_lp))
dholdout = xgb.DMatrix(X_holdout, label=y_holdout, feature_names=ENCODED_COLUMNS)
challenger_holdout_probability = frozen_challenger.predict(dholdout)
assert len(primary_holdout_probability) == len(challenger_holdout_probability) == len(y_holdout) == 6000
assert np.isfinite(primary_holdout_probability).all() and np.isfinite(challenger_holdout_probability).all()

primary_holdout_auc = float(roc_auc_score(y_holdout, primary_holdout_probability))
primary_holdout_ks, primary_holdout_ks_threshold = ks_metric(y_holdout, primary_holdout_probability)
challenger_holdout_auc = float(roc_auc_score(y_holdout, challenger_holdout_probability))
challenger_holdout_ks, challenger_holdout_ks_threshold = ks_metric(y_holdout, challenger_holdout_probability)
auc_difference = challenger_holdout_auc - primary_holdout_auc
ks_difference = challenger_holdout_ks - primary_holdout_ks

holdout_comparison = pd.DataFrame([
    {"Model": "Frozen 10-variable WOE Logistic Primary", "Holdout N": 6000, "AUC": primary_holdout_auc, "KS": primary_holdout_ks, "KS threshold (diagnostic only)": primary_holdout_ks_threshold},
    {"Model": "Frozen 26-variable XGBoost Challenger", "Holdout N": 6000, "AUC": challenger_holdout_auc, "KS": challenger_holdout_ks, "KS threshold (diagnostic only)": challenger_holdout_ks_threshold},
])
display(holdout_comparison)
display(pd.DataFrame({"Value": [auc_difference, ks_difference]}, index=["Challenger - Primary AUC", "Challenger - Primary KS"]))


,Model,Holdout N,AUC,KS,KS threshold (diagnostic only)
0,Frozen 10-variable WOE Logistic Primary,6000,0.758535259474,0.405185813870,0.189626583238
1,Frozen 26-variable XGBoost Challenger,6000,0.779242005131,0.421655549501,0.241897761822


,Value
Challenger - Primary AUC,0.020706745657
Challenger - Primary KS,0.016469735631


## 6. Final interpretation and hard boundary

The comparison below characterizes the observed unseen-Holdout discrimination difference alongside the material interpretability and complexity tradeoff. The frozen WOE Logistic remains the interpretable benchmark regardless of the Challenger result. No post-Holdout tuning, feature change, round change, threshold optimization, calibration, DeLong test, SHAP workstream, or Stage 6 work is performed.


In [7]:
if auc_difference >= 0.01:
    improvement_description = "a clear Holdout AUC improvement"
elif auc_difference > 0.002:
    improvement_description = "a modest Holdout AUC improvement"
elif auc_difference >= -0.002:
    improvement_description = "essentially comparable Holdout AUC"
else:
    improvement_description = "lower Holdout AUC"

interpretation = (
    f"The frozen XGBoost Challenger shows {improvement_description} versus the frozen WOE Logistic Primary "
    f"(AUC difference={auc_difference:.12f}; KS difference={ks_difference:.12f}). "
    "This evidence comes from one sealed Holdout comparison and does not justify post-Holdout model changes. "
    "XGBoost uses a more complex nonlinear 26-variable / 29-column architecture, while the ten-variable WOE Logistic "
    "retains substantially stronger transparency, directional interpretation, and benchmark value."
)
print(interpretation)

stage5_holdout_result = {
    "status": "FINAL_HOLDOUT_COMPARISON_COMPLETE",
    "freeze_gate_status": FREEZE_GATE_STATUS,
    "primary_frozen_status": PRIMARY_FROZEN_STATUS,
    "challenger_frozen_status": CHALLENGER_FROZEN_STATUS,
    "holdout_opening_status": HOLDOUT_OPENING_STATUS,
    "development_rows": 24000,
    "holdout_rows": 6000,
    "train_hash": train_hash,
    "test_hash": test_hash,
    "development_matrix_shape": [24000, 29],
    "holdout_matrix_shape": [6000, 29],
    "final_hyperparameters": FINAL_HYPERPARAMETERS,
    "final_best_round": FINAL_BEST_ROUND,
    "development_mean_cv_auc": FINAL_DEVELOPMENT_MEAN_AUC,
    "development_auc_sd_sample": FINAL_DEVELOPMENT_AUC_SD,
    "development_mean_cv_ks": FINAL_DEVELOPMENT_MEAN_KS,
    "development_ks_sd_sample": FINAL_DEVELOPMENT_KS_SD,
    "primary_holdout_auc": primary_holdout_auc,
    "primary_holdout_ks": primary_holdout_ks,
    "challenger_holdout_auc": challenger_holdout_auc,
    "challenger_holdout_ks": challenger_holdout_ks,
    "challenger_minus_primary_auc": auc_difference,
    "challenger_minus_primary_ks": ks_difference,
    "interpretation": interpretation,
    "post_holdout_model_changes": False,
    "tuning_in_notebook_12": False,
}
print("STAGE5_HOLDOUT_RESULT_JSON=" + json.dumps(stage5_holdout_result, ensure_ascii=False, sort_keys=True, default=float))
print("PASS — one-time same-6,000-row Holdout comparison complete; no post-Holdout decision or tuning occurred.")


The frozen XGBoost Challenger shows a clear Holdout AUC improvement versus the frozen WOE Logistic Primary (AUC difference=0.020706745657; KS difference=0.016469735631). This evidence comes from one sealed Holdout comparison and does not justify post-Holdout model changes. XGBoost uses a more complex nonlinear 26-variable / 29-column architecture, while the ten-variable WOE Logistic retains substantially stronger transparency, directional interpretation, and benchmark value.
STAGE5_HOLDOUT_RESULT_JSON={"challenger_frozen_status": "PASS — frozen Challenger refit on all 24,000 Development rows for exactly 54 rounds without early stopping", "challenger_holdout_auc": 0.7792420051310491, "challenger_holdout_ks": 0.4216555495010459, "challenger_minus_primary_auc": 0.020706745657322778, "challenger_minus_primary_ks": 0.01646973563115145, "development_auc_sd_sample": 0.003556829920052, "development_ks_sd_sample": 0.009842401657479, "development_matrix_shape": [24000, 29], "development_mean_cv_